<a href="https://colab.research.google.com/github/cedo-cloud/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Rainfall Analysis for Kenyan Counties**

This notebook cleans raw rainfall data, analyzes patterns using NumPy, and generates a planting advisory for an NGO.

In [9]:
import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ", "9", "12", "", "45", "182", "96"]

##**Data Cleaning**

The Machakos rainfall dataset contains missing and invalid entries. The following function cleans the data by converting valid values into integers and replacing invalid entries with the average of the available valid values.

In [10]:
def clean_machakos(raw_data):
    valid_values = []
    imputed_months = []

    for i, val in enumerate(raw_data):
        try:
            clean_val = int(val)
            valid_values.append((i, clean_val))
        except ValueError:
            imputed_months.append(months[i])

    valid_nums = [v[1] for v in valid_values]
    avg_val = round(sum(valid_nums) / len(valid_nums))

    cleaned_row = []
    valid_indices = {v[0]: v[1] for v in valid_values}

    for i in range(12):
        if i in valid_indices:
            cleaned_row.append(valid_indices[i])
        else:
            cleaned_row.append(avg_val)

    print(f"Imputed months for Machakos: {', '.join(imputed_months)} with value {avg_val} mm")

    return cleaned_row


machakos_clean = clean_machakos(machakos_raw)

Imputed months for Machakos: April, September with value 68 mm


## **Creating the NumPy Array**

Combine the cleaned rainfall records for all five counties into a single NumPy array. Display the array's shape and data type to verify its structure.

In [11]:
all_rows = clean_rows + [machakos_clean]
rainfall_data = np.array(all_rows)

print(f"Array shape: {rainfall_data.shape}")
print(f"Array dtype: {rainfall_data.dtype}")

Array shape: (5, 12)
Array dtype: int64


## **Annual Rainfall Totals and Monthly Averages**

Calculate the total annual rainfall for each county and the average rainfall across all counties for each month.

In [12]:
annual_totals = np.sum(rainfall_data, axis=1)
monthly_averages = np.mean(rainfall_data, axis=0)

print("Annual totals per county:", annual_totals)
print("Monthly averages across counties:", np.round(monthly_averages, 2))

Annual totals per county: [1045  916 1371 1075  811]
Monthly averages across counties: [ 47.   47.6  91.  164.4 171.2  67.6  56.6  58.2  64.2  67.8 127.4  80.6]


## **Identifying the Wettest Month**

Use NumPy's argmax() function to identify the month with the highest rainfall for each county.

In [13]:
wettest_month_indices = np.argmax(rainfall_data, axis=1)

print("Wettest month per county:")

for county, idx in zip(counties, wettest_month_indices):
    print(f"- {county}: {months[idx]}")

Wettest month per county:
- Nairobi: April
- Nakuru: April
- Kisumu: April
- Mombasa: May
- Machakos: November


## **Identifying Dry Months**

Identify months receiving less than 50 mm of rainfall and calculate the number of dry months experienced by each county.

In [14]:
dry_mask = rainfall_data < 50
dry_month_counts = np.sum(dry_mask, axis=1)

print("Dry months (<50mm) count per county:", dry_month_counts)

Dry months (<50mm) count per county: [4 3 0 2 6]


## **Recommended Planting Windows**

Calculate consecutive two-month rainfall totals to identify the best planting window for each county. The window with the highest combined rainfall is recommended.

In [15]:
consecutive_rainfall = rainfall_data[:, :-1] + rainfall_data[:, 1:]
best_window_indices = np.argmax(consecutive_rainfall, axis=1)

print("Recommended planting windows:")

for county, idx in zip(counties, best_window_indices):
    window = f"{months[idx]}-{months[idx+1]}"
    print(f"- {county}: {window}")

Recommended planting windows:
- Nairobi: April-May
- Nakuru: April-May
- Kisumu: April-May
- Mombasa: April-May
- Machakos: November-December


## **Exporting the Analysis Results**

Export the rainfall analysis results into a CSV file named `rainfall_summary.csv`. The file contains each county's annual rainfall total, wettest month, number of dry months, and recommended planting window.

In [16]:
with open('rainfall_summary.csv', 'w', newline='') as f:
    writer = csv.writer(f)

    writer.writerow([
        'county',
        'annual_total_mm',
        'wettest_month',
        'dry_months',
        'planting_window'
    ])

    for i, county in enumerate(counties):
        w_idx = wettest_month_indices[i]
        p_idx = best_window_indices[i]

        writer.writerow([
            county,
            int(annual_totals[i]),
            months[w_idx],
            int(dry_month_counts[i]),
            f"{months[p_idx]}-{months[p_idx+1]}"
        ])

print("rainfall_summary.csv written successfully.")

rainfall_summary.csv written successfully.


## **Planting Advisory**

Based on the rainfall analysis, farmers in Nairobi, Nakuru, Kisumu, and Mombasa should target the April to May window for planting, as these months consistently provide the highest consecutive rainfall. Conversely, Machakos experiences its peak moisture availability later in the year, making November to December the optimal planting period for that region.

Stakeholders should note that the Machakos dataset contained missing and malformed entries that were imputed using the average of valid months. Therefore, while the overall trends are reliable, local verification of current soil moisture is strongly advised before making critical agricultural investments.

## **Data Quality Notes**

- The Machakos raw data contained invalid entries ("N/A" and empty strings) for April and September.
- These missing values were imputed with the rounded average (68 mm) of the remaining 10 valid months.
- All other counties provided clean, numeric data, ensuring high confidence in their respective totals and planting window recommendations.